# APUSH FRQ Grader v5 — final Colab run

Merge v4, train separate score and feedback adapters, compare on private development data, freeze configuration, and evaluate once on the 53-case CB-derived golden set. Private rows and per-case predictions must not be redistributed. Because capped golden excerpts informed synthetic style, the golden evaluation is **development-informed**, not untouched.

## 1. GPU, frozen code, dependencies, and Drive

In [1]:
import os, sys, json, hashlib, subprocess, zipfile
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Attach a GPU runtime.'
REPO=Path('/content/slm'); REF=os.environ.get('SLM_GIT_REF','main')
if not (REPO/'.git').exists(): subprocess.run(['git','clone','https://github.com/aryanjverma/slm.git',str(REPO)],check=True)
subprocess.run(['git','fetch','--prune','origin',REF],cwd=REPO,check=True)
subprocess.run(['git','checkout','--detach','FETCH_HEAD'],cwd=REPO,check=True)
REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
os.chdir(REPO)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[train]','sentencepiece','tiktoken'],check=True)
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/slm_v5')
PRIVATE=Path(os.environ.get('V5_PRIVATE_ROOT',str(ROOT/'private_repo/artifacts/data/v5/private')))
V4=Path(os.environ.get('V4_ADAPTER_DIR','/content/drive/MyDrive/slm_v4/apush-frq-grader-v4-assistant-only-r1/final'))
MERGED=ROOT/'merged_v4_base'; SCORER_RUN=ROOT/'scorer/final-r1'; FEEDBACK_RUN=ROOT/'feedback/final-r1'; EVAL=ROOT/'eval/final-r1'
for path in (ROOT,EVAL): path.mkdir(parents=True,exist_ok=True)
print(torch.cuda.get_device_name(0),REVISION)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Tesla T4 6e66e4e67cb0c1e4b6da15376c79b7240fc2259b


In [2]:
from pathlib import Path

PRIVATE = Path(
    "/content/drive/MyDrive/slm_v5/private_repo/private"
)

print(PRIVATE.exists())
print([path.name for path in PRIVATE.glob("*")])

True
['logs', 'judged_r2', 'audits', 'validation_audit_v5.json', 'production_hard_gate_audit_r2.json', 'manual_review_packet_RESET.json', 'pilot_approval_v5.json', 'validation_audit_r2.json', 'rejected_essays_r2.jsonl', 'pilot_essays_v5.jsonl', 'private_use_manifest_v5.json', 'pilot_review_decisions_v5.json', 'pilot_hard_gate_audit_v5.json', 'r1_corpus_discarded.json', 'manual_review_packet_v5.jsonl', 'manual_review_approval_v5.json', 'dev_chat_v5_scorer.jsonl', 'dev_chat_v5_feedback.jsonl', 'replay_cases_v4_for_v5.jsonl', 'dev_cases_v5.jsonl', 'assembly_audit_v5.json', 'judged', 'raw_essays', 'pilot_raw', 'raw_essays_r2', 'selected_cases_v5_provisional.jsonl', 'external_candidates_v5.jsonl', 'accepted_essays_r2.jsonl', 'train_cases_v5.jsonl', 'validated_candidates_r2.jsonl', 'train_chat_v5_scorer.jsonl', 'train_chat_v5_feedback.jsonl', 'train_cases_v5_with_replay.jsonl', 'validated_candidates_v5.jsonl']


In [3]:
import importlib, subprocess, sys
from pathlib import Path

REPO = Path("/content/slm")

subprocess.run(
    ["git", "fetch", "--prune", "origin", "main"],
    cwd=REPO,
    check=True,
)
subprocess.run(
    ["git", "checkout", "--detach", "FETCH_HEAD"],
    cwd=REPO,
    check=True,
)

source = REPO / "src/apush_frq_grader_slm/training_v5.py"
assert "def validate_v5_training_preflight(" in source.read_text(), (
    "Checked-out source still lacks preflight"
)

for name in list(sys.modules):
    if name == "apush_frq_grader_slm" or name.startswith("apush_frq_grader_slm."):
        del sys.modules[name]

sys.path[:] = [p for p in sys.path if p != str(REPO / "src")]
sys.path.insert(0, str(REPO / "src"))
importlib.invalidate_caches()

from apush_frq_grader_slm.training_v5 import validate_v5_training_preflight

preflight = validate_v5_training_preflight(
    PRIVATE,
    data_path=PRIVATE / "train_cases_v5_with_replay.jsonl",
    golden_cases_path=REPO / "artifacts/data/eval_cb_cases.jsonl",
)
print(preflight)


{'approved': True, 'review_packet_sha256': 'b5ae4d76339c5d26986f93d655eff2abafc0887f5d092293b7127679e1498321', 'combined_training_sha256': '0de79600ad2a4479732c74a485986e03b7ca8ea1a4285268ee1765834a0c23ae', 'training_rows': 615, 'development_rows': 60, 'golden_eval_rows_in_training': 0}


## 2. Private preflight and approved 615-row runtime set

The finalized private directory contains 540 new train cases, 60 development cases, and 75 balanced v4 replay cases. Approval is bound to the reviewed 60-case packet. This cell verifies counts and receipts, then creates a private Drive-only combined training JSONL; it never prints or embeds rows.

In [4]:
MANIFEST=PRIVATE/'private_use_manifest_v5.json'
AUDIT=PRIVATE/'assembly_audit_v5.json'
APPROVAL=PRIVATE/'manual_review_approval_v5.json'
PACKET=PRIVATE/'manual_review_packet_v5.jsonl'
NEW_TRAIN=PRIVATE/'train_cases_v5.jsonl'
DEV=PRIVATE/'dev_cases_v5.jsonl'
REPLAY=PRIVATE/'replay_cases_v4_for_v5.jsonl'
TRAIN=PRIVATE/'train_cases_v5_with_replay.jsonl'
GOLDEN=REPO/'artifacts/data/eval_cb_cases.jsonl'
# Colab may not refresh editable installs in the current kernel; prefer this checkout explicitly.
src_path=str(REPO/'src')
if src_path not in sys.path: sys.path.insert(0,src_path)
from apush_frq_grader_slm.training_v5 import validate_v5_training_preflight
preflight=validate_v5_training_preflight(PRIVATE,data_path=TRAIN,golden_cases_path=GOLDEN)
assert preflight['training_rows']==615 and preflight['golden_eval_rows_in_training']==0
print(preflight)

{'approved': True, 'review_packet_sha256': 'b5ae4d76339c5d26986f93d655eff2abafc0887f5d092293b7127679e1498321', 'combined_training_sha256': '0de79600ad2a4479732c74a485986e03b7ca8ea1a4285268ee1765834a0c23ae', 'training_rows': 615, 'development_rows': 60, 'golden_eval_rows_in_training': 0}


## 3. Merge the completed v4 adapter in FP16

The inherited-base receipt hashes both the source adapter and merged model. Reruns skip a complete merge.

In [5]:
BASE='Qwen/Qwen2.5-0.5B-Instruct'
MERGE=REPO/'scripts/merge_v4_adapter.py'; MERGED_META=MERGED/'v5_inherited_base.json'
assert MERGE.exists() and (V4/'adapter_config.json').exists()
if not MERGED_META.exists():
 subprocess.run([sys.executable,str(MERGE),'--v4-adapter',str(V4),'--base-model',BASE,'--output',str(MERGED),'--no-bf16'],check=True)
merged_meta=json.loads(MERGED_META.read_text(encoding='utf-8'))
assert merged_meta['v4_adapter_sha256'] and merged_meta['inherited_base_sha256']
print('Inherited base:',merged_meta['inherited_base_sha256'])

Inherited base: e432cb6a40d471fd0667228a4e89e52895b8dfba26b390f07b1c69536626c2bd


## 4. Train scorer and score-conditioned feedback adapters

Both are fresh rank-16 QLoRA adapters with 4,096-token context and resume support. The scorer weights numeric tokens 4x and uses four epochs at 1e-4. Feedback uses two epochs at 5e-5. Checkpoints are evaluated every 100 steps with patience two.

In [6]:
COMMON={'max_seq_length':4096,'lora_rank':16,'batch_size':1,'grad_accum':4,'warmup_ratio':.03,'eval_steps':100,'early_stopping_patience':2,'seed':13}
CONFIG={'scorer':{'epochs':4,'learning_rate':1e-4,'score_token_weight':4.0},'feedback':{'epochs':2,'learning_rate':5e-5}}
TRAIN_SCRIPT=REPO/'scripts/train_v5.py'
def latest(run):
 good=[x for x in run.glob('checkpoint-*') if (x/'trainer_state.json').exists()]
 return max(good,key=lambda x:int(x.name.rsplit('-',1)[-1])) if good else None
def train(task,run):
 final=run/'final'
 if (final/'adapter_config.json').exists(): return
 cfg=CONFIG[task]
 cmd=[sys.executable,str(TRAIN_SCRIPT),'--task',task,'--model',str(MERGED),'--data',str(TRAIN),'--eval-data',str(DEV),'--private-dir',str(PRIVATE),'--golden-cases',str(GOLDEN),'--output',str(run),'--max-seq-length',str(COMMON['max_seq_length']),'--lora-rank',str(COMMON['lora_rank']),'--batch-size','1','--grad-accum','4','--warmup-ratio','.03','--eval-steps','100','--early-stopping-patience','2','--seed','13','--epochs',str(cfg['epochs']),'--learning-rate',str(cfg['learning_rate'])]
 if task=='scorer': cmd += ['--score-token-weight','4']
 checkpoint=latest(run)
 if checkpoint: cmd += ['--resume-from-checkpoint',str(checkpoint)]
 log=ROOT/f'v5_{task}_final-r1.log'
 with log.open('a',encoding='utf-8') as stream:
  proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
  for line in proc.stdout: print(line,end=''); stream.write(line); stream.flush()
  assert proc.wait()==0,f'{task} failed; inspect {log}'
train('scorer',SCORER_RUN); train('feedback',FEEDBACK_RUN)
SCORER=SCORER_RUN/'final'; FEEDBACK=FEEDBACK_RUN/'final'
for path in (SCORER,FEEDBACK):
 assert (path/'adapter_config.json').exists() and (path/'v5_training_metadata.json').exists()

## 5. Select checkpoints, then run a ten-case two-pass smoke evaluation

Every saved scorer and feedback checkpoint is first evaluated on the 60-case development set. Monitored subprocesses report phase boundaries, completed rows, elapsed time, and GPU utilization, and stop after 20 minutes without progress. The final ten-case smoke evaluation exercises validated scores, deterministic totals, grounded feedback, and fallback feedback without printing private cases.

In [9]:
PACKAGE=REPO/'scripts/package_v5_bundle.py'; GRADE_SCRIPT=REPO/'scripts/grade_v5.py'; EVAL_SCRIPT=REPO/'scripts/eval_v5.py'
assert PACKAGE.exists() and GRADE_SCRIPT.exists() and EVAL_SCRIPT.exists()
import time
from datetime import datetime
HEARTBEAT_SECONDS=30; STALL_MINUTES=20
def completed_rows(path):
 if path is None or not path.exists(): return 0
 with path.open(encoding='utf-8') as stream: return sum(bool(line.strip()) for line in stream)
def gpu_status():
 try:
  return subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total','--format=csv,noheader,nounits'],text=True,timeout=10).strip()
 except Exception as exc: return f'unavailable ({type(exc).__name__})'
def monitored_run(cmd,label,results=None,stall_minutes=STALL_MINUTES):
 started=time.monotonic(); last_progress=started; rows=completed_rows(results)
 env=os.environ.copy(); env['PYTHONUNBUFFERED']='1'
 print(f'\n[{datetime.now().isoformat(timespec="seconds")}] START {label}; existing rows={rows}',flush=True)
 proc=subprocess.Popen(cmd,env=env)
 try:
  while True:
   try:
    returncode=proc.wait(timeout=HEARTBEAT_SECONDS); break
   except subprocess.TimeoutExpired:
    current=completed_rows(results)
    if current>rows: rows=current; last_progress=time.monotonic()
    elapsed=(time.monotonic()-started)/60; idle=(time.monotonic()-last_progress)/60
    print(f'[{datetime.now().isoformat(timespec="seconds")}] HEARTBEAT {label}: pid={proc.pid}, elapsed={elapsed:.1f}m, rows={rows}, idle={idle:.1f}m, GPU={gpu_status()}',flush=True)
    if idle>=stall_minutes: raise TimeoutError(f'{label} stalled with no completed-row progress for {idle:.1f} minutes')
 except BaseException:
  if proc.poll() is None:
   print(f'STOPPING {label} subprocess {proc.pid}; completed rows remain resumable',flush=True); proc.terminate()
   try: proc.wait(timeout=30)
   except subprocess.TimeoutExpired: proc.kill(); proc.wait()
  raise
 if returncode: raise subprocess.CalledProcessError(returncode,cmd)
 print(f'[{datetime.now().isoformat(timespec="seconds")}] DONE {label} in {(time.monotonic()-started)/60:.1f}m; rows={completed_rows(results)}',flush=True)
def adapter_candidates(run):
 paths=sorted((p for p in run.glob('checkpoint-*') if (p/'adapter_config.json').exists()),key=lambda p:int(p.name.rsplit('-',1)[-1]))
 paths.append(run/'final')
 return list(dict.fromkeys(path.resolve() for path in paths))
def temporary_eval(label,scorer,feedback):
 bundle=ROOT/'checkpoint_selection/bundles'/label; out=ROOT/'checkpoint_selection/eval'/label
 results=out/f'{label}_real_results.jsonl'
 monitored_run([sys.executable,'-u',str(PACKAGE),'--bundle',str(bundle),'--inherited-base',str(MERGED),'--scorer',str(scorer),'--feedback',str(feedback),'--reference-only','--overwrite'],f'{label} package',stall_minutes=5)
 monitored_run([sys.executable,'-u',str(EVAL_SCRIPT),'--bundle',str(bundle),'--eval-path',str(DEV),'--output-dir',str(out),'--model-name',label,'--bootstrap-samples','100','--no-verify-hashes','--resume'],f'{label} evaluation',results)
 return json.loads((out/f'{label}_real_summary.json').read_text(encoding='utf-8'))
scorer_candidates=adapter_candidates(SCORER_RUN); feedback_candidates=adapter_candidates(FEEDBACK_RUN)
print(f'Checkpoint selection plan: {len(scorer_candidates)} scorer evaluations + {len(feedback_candidates)} feedback evaluations; each has 60 two-pass cases.',flush=True)
scorer_rank=[]
for index,path in enumerate(scorer_candidates):
 summary=temporary_eval(f'scorer-{index:02d}',path,FEEDBACK)
 scorer_rank.append({'adapter':str(path),'summary':summary})
def scorer_key(item):
 s=item['summary']; qwk=s['qwk'] if s['qwk'] is not None else -1.0; exact=s['criterion_exact_rates']
 return (qwk,-s['total_mae'],exact['evidence'],sum(exact.values())/len(exact))
scorer_rank.sort(key=scorer_key,reverse=True); SCORER=Path(scorer_rank[0]['adapter'])
feedback_rank=[]
for index,path in enumerate(feedback_candidates):
 summary=temporary_eval(f'feedback-{index:02d}',SCORER,path)
 feedback_rank.append({'adapter':str(path),'summary':summary})
def feedback_key(item):
 s=item['summary']; return (s['evidence_grounding_rate'],s['structured_output_valid_rate'],-s['feedback_fallback_rate'])
feedback_rank.sort(key=feedback_key,reverse=True); FEEDBACK=Path(feedback_rank[0]['adapter'])
SELECTION=ROOT/'checkpoint_selection/selected.json'; SELECTION.parent.mkdir(parents=True,exist_ok=True)
SELECTION.write_text(json.dumps({'scorer':scorer_rank,'feedback':feedback_rank,'selected_scorer':str(SCORER),'selected_feedback':str(FEEDBACK)},indent=2)+'\n',encoding='utf-8')
BUNDLE=ROOT/'bundle/final-r1'
if not (BUNDLE/'v5_bundle.json').exists(): monitored_run([sys.executable,'-u',str(PACKAGE),'--bundle',str(BUNDLE),'--inherited-base',str(MERGED),'--scorer',str(SCORER),'--feedback',str(FEEDBACK),'--overwrite'],'final bundle package',stall_minutes=5)
assert (BUNDLE/'v5_bundle.json').exists()
SMOKE=EVAL/'smoke'; SMOKE_NAME='apush-frq-grader-v5-smoke'
SMOKE_RESULTS=SMOKE/f'{SMOKE_NAME}_real_results.jsonl'
monitored_run([sys.executable,'-u',str(EVAL_SCRIPT),'--bundle',str(BUNDLE),'--eval-path',str(DEV),'--output-dir',str(SMOKE),'--model-name',SMOKE_NAME,'--limit','10','--bootstrap-samples','100','--resume'],'final ten-case smoke evaluation',SMOKE_RESULTS)
SMOKE_SUMMARY=SMOKE/f'{SMOKE_NAME}_real_summary.json'
smoke=json.loads(SMOKE_SUMMARY.read_text(encoding='utf-8'))
assert smoke['count']==10 and smoke['structured_output_valid_rate']==1.0 and smoke['deterministic_total_rate']==1.0
print({k:smoke[k] for k in ('count','structured_output_valid_rate','deterministic_total_rate','feedback_fallback_rate')})

Checkpoint selection plan: 4 scorer evaluations + 4 feedback evaluations; each has 60 two-pass cases.

[2026-07-13T00:38:10] START scorer-00 package; existing rows=0


KeyboardInterrupt: 

In [10]:
from pathlib import Path
from datetime import datetime
import time

ROOT = Path("/content/drive/MyDrive/slm_v5")

def row_count(path):
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

def candidates(run_dir):
    checkpoints = sorted(
        (
            p for p in run_dir.glob("checkpoint-*")
            if (p / "adapter_config.json").exists()
        ),
        key=lambda p: int(p.name.rsplit("-", 1)[-1]),
    )
    final = run_dir / "final"
    if (final / "adapter_config.json").exists():
        checkpoints.append(final)
    return checkpoints

scorers = candidates(ROOT / "scorer/final-r1")
feedback = candidates(ROOT / "feedback/final-r1")
eval_root = ROOT / "checkpoint_selection/eval"

jobs = (
    [(f"scorer-{i:02d}", 60) for i in range(len(scorers))]
    + [(f"feedback-{i:02d}", 60) for i in range(len(feedback))]
)

completed = 0
total = sum(limit for _, limit in jobs) + 10

print(f"Scorer candidates: {len(scorers)}")
print(f"Feedback candidates: {len(feedback)}\n")

for label, limit in jobs:
    results = eval_root / label / f"{label}_real_results.jsonl"
    summary = eval_root / label / f"{label}_real_summary.json"
    count = min(row_count(results), limit)
    completed += count

    if summary.exists():
        status = "DONE"
    elif count:
        age = time.time() - results.stat().st_mtime
        status = f"RUNNING/PAUSED — last update {age/60:.1f} min ago"
    else:
        status = "PENDING"

    print(f"{label:12} {count:2}/{limit}  {status}")

smoke_results = (
    ROOT / "eval/final-r1/smoke"
    / "apush-frq-grader-v5-smoke_real_results.jsonl"
)
smoke_summary = smoke_results.with_name(
    "apush-frq-grader-v5-smoke_real_summary.json"
)
smoke_count = min(row_count(smoke_results), 10)
completed += smoke_count

status = "DONE" if smoke_summary.exists() else (
    "RUNNING/PAUSED" if smoke_count else "PENDING"
)
print(f"\n{'final smoke':12} {smoke_count:2}/10  {status}")
print(f"\nApproximate overall progress: {completed}/{total} "
    f"({100 * completed / total:.1f}%)")
print("Checkpoint selection complete:",
    (ROOT / "checkpoint_selection/selected.json").exists())

Scorer candidates: 4
Feedback candidates: 4

scorer-00    60/60  DONE
scorer-01    60/60  DONE
scorer-02    60/60  DONE
scorer-03    60/60  DONE
feedback-00  60/60  DONE
feedback-01  60/60  DONE
feedback-02  60/60  DONE
feedback-03  60/60  DONE

final smoke  10/10  DONE

Approximate overall progress: 490/490 (100.0%)
Checkpoint selection complete: True


## 6. Private development comparison, then freeze

Compare prompted base, v4, and v5 on all 60 private development cases. Review only aggregate results. Freezing binds code, datasets, inherited base, adapters, prompts, and hyperparameters; golden evaluation remains blocked until then.

In [12]:
DEV_OUT=EVAL/'development'; LEGACY_EVAL=REPO/'scripts/eval_hf_model.py'; V5_DEV_NAME='apush-frq-grader-v5-development'
BUNDLE=ROOT/'bundle/final-r1'
assert (BUNDLE/'v5_bundle.json').exists(),'Final bundle is missing; complete section 5 first.'
BASE_DEV_RESULTS=DEV_OUT/'base/qwen-base-v5-dev_real_results.jsonl'
V4_DEV_RESULTS=DEV_OUT/'v4/v4-v5-dev_real_results.jsonl'
V5_DEV_RESULTS=DEV_OUT/'v5'/f'{V5_DEV_NAME}_real_results.jsonl'
monitored_run([sys.executable,'-u',str(LEGACY_EVAL),'--model',BASE,'--model-name','qwen-base-v5-dev','--eval-path',str(DEV),'--output-dir',str(DEV_OUT/'base'),'--prompt-version','v4','--real-eval','--resume'],'development base evaluation',BASE_DEV_RESULTS)
monitored_run([sys.executable,'-u',str(LEGACY_EVAL),'--model',str(V4),'--model-name','v4-v5-dev','--eval-path',str(DEV),'--output-dir',str(DEV_OUT/'v4'),'--prompt-version','v4','--real-eval','--resume'],'development v4 evaluation',V4_DEV_RESULTS)
monitored_run([sys.executable,'-u',str(EVAL_SCRIPT),'--bundle',str(BUNDLE),'--eval-path',str(DEV),'--output-dir',str(DEV_OUT/'v5'),'--model-name',V5_DEV_NAME,'--bootstrap-samples','500','--resume'],'development v5 evaluation',V5_DEV_RESULTS)
def first_jsonl(path): return json.loads(path.read_text(encoding='utf-8').splitlines()[0])
comparison={'count':60,'base':first_jsonl(DEV_OUT/'base/qwen-base-v5-dev_real_summary.jsonl'),'v4':first_jsonl(DEV_OUT/'v4/v4-v5-dev_real_summary.jsonl'),'v5':json.loads((DEV_OUT/'v5'/f'{V5_DEV_NAME}_real_summary.json').read_text(encoding='utf-8'))}
DEV_SUMMARY=DEV_OUT/'comparison_summary.json'; DEV_SUMMARY.write_text(json.dumps(comparison,indent=2)+'\n',encoding='utf-8')
assert all(model['count']==60 for model in comparison.values() if isinstance(model,dict))
print(json.dumps(comparison,indent=2))


[2026-07-13T00:39:47] START development base evaluation; existing rows=60


[2026-07-13T00:40:03] DONE development base evaluation in 0.3m; rows=60

[2026-07-13T00:40:03] START development v4 evaluation; existing rows=60
[2026-07-13T00:40:18] DONE development v4 evaluation in 0.3m; rows=60

[2026-07-13T00:40:18] START development v5 evaluation; existing rows=55
[2026-07-13T00:40:48] HEARTBEAT development v5 evaluation: pid=64518, elapsed=0.5m, rows=56, idle=0.0m, GPU=25, 1667, 15360
[2026-07-13T00:41:18] HEARTBEAT development v5 evaluation: pid=64518, elapsed=1.0m, rows=59, idle=0.0m, GPU=90, 2845, 15360
[2026-07-13T00:41:48] HEARTBEAT development v5 evaluation: pid=64518, elapsed=1.5m, rows=61, idle=0.0m, GPU=99, 2845, 15360
[2026-07-13T00:42:18] HEARTBEAT development v5 evaluation: pid=64518, elapsed=2.0m, rows=64, idle=0.0m, GPU=37, 1377, 15360
[2026-07-13T00:42:20] DONE development v5 evaluation in 2.0m; rows=65
{
  "count": 60,
  "base": {
    "model_name": "qwen-base-v5-dev",
    "count": 60,
    "structured_output_valid_rate": 0.0167,
    "exact_match_r

In [14]:
FREEZE_CONFIGURATION=True
import hashlib
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
TRAIN=PRIVATE/'train_cases_v5_with_replay.jsonl'; DEV=PRIVATE/'dev_cases_v5.jsonl'; APPROVAL=PRIVATE/'manual_review_approval_v5.json'
MERGED_META=ROOT/'merged_v4_base/v5_inherited_base.json'; merged_meta=json.loads(MERGED_META.read_text(encoding='utf-8'))
COMMON={'max_seq_length':4096,'lora_rank':16,'batch_size':1,'grad_accum':4,'warmup_ratio':.03,'eval_steps':100,'early_stopping_patience':2,'seed':13}
CONFIG={'scorer':{'epochs':4,'learning_rate':1e-4,'score_token_weight':4.0},'feedback':{'epochs':2,'learning_rate':5e-5}}
BUNDLE=ROOT/'bundle/final-r1'; DEV_SUMMARY=ROOT/'eval/final-r1/development/comparison_summary.json'
SELECTION=ROOT/'checkpoint_selection/selected.json'
selection=json.loads(SELECTION.read_text(encoding='utf-8'))
SCORER=Path(selection['selected_scorer']); FEEDBACK=Path(selection['selected_feedback'])
FROZEN=ROOT/'v5_frozen_config_final-r1.json'
def adapter_weight(path):
 files=list(path.glob('adapter_model.*')); assert len(files)==1,path
 return files[0]
frozen={'run_id':'final-r1','repository_revision':REVISION,'train_sha256':sha(TRAIN),'dev_sha256':sha(DEV),'approval_sha256':sha(APPROVAL),'inherited_base_sha256':merged_meta['inherited_base_sha256'],'scorer_adapter_sha256':sha(adapter_weight(SCORER)),'feedback_adapter_sha256':sha(adapter_weight(FEEDBACK)),'checkpoint_selection_sha256':sha(SELECTION),'bundle_manifest_sha256':sha(BUNDLE/'v5_bundle.json'),'development_summary_sha256':sha(DEV_SUMMARY),'common':COMMON,'tasks':CONFIG,'development_informed':True}
if FREEZE_CONFIGURATION:
 if FROZEN.exists(): assert json.loads(FROZEN.read_text(encoding='utf-8'))==frozen,'Changed configuration needs a new run id'
 else: FROZEN.write_text(json.dumps(frozen,indent=2)+'\n',encoding='utf-8')
assert FROZEN.exists(),'Freeze the reviewed development configuration first.'
print('Frozen:',FROZEN)

Frozen: /content/drive/MyDrive/slm_v5/v5_frozen_config_final-r1.json


## 7. One-time 53-case golden evaluation

Run only after freezing. It must emit aggregate criterion confusion matrices, score distributions, length/reference-total calibration, evidence 0/1/2 errors, and bootstrap confidence intervals. Never retune against these answers.

In [16]:
EVAL=ROOT/'eval/final-r1'; EVAL_SCRIPT=REPO/'scripts/eval_v5.py'; BUNDLE=ROOT/'bundle/final-r1'
GOLD=EVAL/'golden'; GOLD_NAME='apush-frq-grader-v5-final'; FROZEN_CONFIG=ROOT/'v5_frozen_config_final-r1.json'
assert FROZEN_CONFIG.exists(),'Freeze the reviewed development configuration first.'
GOLD_SUMMARY=GOLD/f'{GOLD_NAME}_real_summary.json'
GOLD_RESULTS=GOLD/f'{GOLD_NAME}_real_results.jsonl'
monitored_run([sys.executable,'-u',str(EVAL_SCRIPT),'--bundle',str(BUNDLE),'--eval-path',str(REPO/'artifacts/data/eval_cb_cases.jsonl'),'--output-dir',str(GOLD),'--model-name',GOLD_NAME,'--development-informed','--bootstrap-samples','2000','--resume'],'one-time 53-case golden evaluation',GOLD_RESULTS)
golden=json.loads(GOLD_SUMMARY.read_text(encoding='utf-8')); gold=golden
assert golden['count']==53 and golden['development_informed'] is True
golden_evaluation_is_development_informed=True
print(json.dumps(golden,indent=2))


[2026-07-13T00:51:55] START one-time 53-case golden evaluation; existing rows=32
[2026-07-13T00:52:25] HEARTBEAT one-time 53-case golden evaluation: pid=67434, elapsed=0.5m, rows=32, idle=0.5m, GPU=33, 1257, 15360
[2026-07-13T00:52:55] HEARTBEAT one-time 53-case golden evaluation: pid=67434, elapsed=1.0m, rows=36, idle=0.0m, GPU=22, 1701, 15360
[2026-07-13T00:53:26] HEARTBEAT one-time 53-case golden evaluation: pid=67434, elapsed=1.5m, rows=38, idle=0.0m, GPU=38, 1701, 15360
[2026-07-13T00:53:56] HEARTBEAT one-time 53-case golden evaluation: pid=67434, elapsed=2.0m, rows=41, idle=0.0m, GPU=30, 1701, 15360
[2026-07-13T00:54:26] HEARTBEAT one-time 53-case golden evaluation: pid=67434, elapsed=2.5m, rows=44, idle=0.0m, GPU=26, 1701, 15360
[2026-07-13T00:54:56] HEARTBEAT one-time 53-case golden evaluation: pid=67434, elapsed=3.0m, rows=47, idle=0.0m, GPU=23, 1701, 15360
[2026-07-13T00:55:26] HEARTBEAT one-time 53-case golden evaluation: pid=67434, elapsed=3.5m, rows=49, idle=0.0m, GPU=35,

## 8. Locked release gates and aggregate-only export

Any failed gate means non-production-ready; do not tune on golden results. The ZIP excludes all JSONL, essays, review packets, prompts/excerpts, and per-case predictions.

In [17]:
RELEASE_SCRIPT=REPO/'scripts/check_v5_release.py'; GATE=GOLD/'release_gate_receipt.json'
release_run=subprocess.run([sys.executable,str(RELEASE_SCRIPT),'--summary',str(GOLD_SUMMARY),'--output',str(GATE)])
decision=json.loads(GATE.read_text(encoding='utf-8'))
assert (release_run.returncode==0)==decision['release_ready']
decision['policy']='All gates passed.' if decision['release_ready'] else 'Non-production-ready; do not retune on golden answers.'
REPORT=ROOT/'reports/final-r1/v5_run_report.md'; REPORT.parent.mkdir(parents=True,exist_ok=True)
REPORT.write_text(f"""# APUSH FRQ Grader v5 Run Report

Golden evaluation: 53 CB-derived cases (**development-informed**). Capped golden style excerpts shaped synthetic generation; this is not untouched.

| Metric | Result |
|---|---:|
| QWK | {golden['qwk']:.4f} |
| Total MAE | {golden['total_mae']:.4f} |
| Within one | {golden['total_within_one_rate']:.2%} |
| Structured validity | {golden['structured_output_valid_rate']:.2%} |
| Feedback grounding | {golden['evidence_grounding_rate']:.2%} |

Release: **{'PASS' if decision['release_ready'] else 'FAIL — non-production-ready'}**. Do not retune against golden answers.
""",encoding='utf-8')
DIAGNOSTICS=GOLD/f'{GOLD_NAME}_v5_diagnostics.json'; ZIP=ROOT/'v5_report_final-r1.zip'
exports=[REPORT,GOLD_SUMMARY,DIAGNOSTICS,GATE,FROZEN_CONFIG,MERGED_META,SELECTION,BUNDLE/'v5_bundle.json']
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as z:
 for path in exports:z.write(path,path.name)
with zipfile.ZipFile(ZIP) as z:names=z.namelist()
assert not any(name.endswith('.jsonl') for name in names)
assert not any(x in name.lower() for name in names for x in ('essay','prediction','review_packet'))
print(json.dumps(decision,indent=2)); print('Aggregate-only bundle:',ZIP)
DOWNLOAD_REPORT_BUNDLE=True
if DOWNLOAD_REPORT_BUNDLE:
 from google.colab import files
 files.download(str(ZIP))

{
  "checks": {
    "all_criteria_improve_over_v4": false,
    "evidence_grounding_rate": true,
    "predicted_total_mean_bias": false,
    "qwk": false,
    "structured_output_valid_rate": true,
    "total_mae": false,
    "total_within_one_rate": false
  },
  "criterion_checks": {
    "analysis_reasoning": true,
    "contextualization": false,
    "evidence": true,
    "thesis": true
  },
  "decision": "non_production_ready",
  "release_ready": false,
  "thresholds": {
    "criterion_exact_strictly_above": {
      "analysis_reasoning": 0.3208,
      "contextualization": 0.3962,
      "evidence": 0.1698,
      "thesis": 0.5283
    },
    "evidence_grounding_rate": 0.85,
    "maximum_mean_bias": 0.5,
    "qwk": 0.4,
    "reference_total_mean": 4.0377,
    "structured_output_valid_rate": 0.98,
    "total_mae": 1.5,
    "total_within_one_rate": 0.6
  },
  "policy": "Non-production-ready; do not retune on golden answers."
}
Aggregate-only bundle: /content/drive/MyDrive/slm_v5/v5_report_fi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>